# Demand Forecasting in Production

Companion notebook for the [Demand Forecasting lesson](https://ml-viz-ruby.vercel.app/courses/time-series/04-demand-forecasting-in-production).

**The idea in one sentence.** Real forecasting systems care about more than a point
prediction: they must be **coherent across a hierarchy** (store → region → national
forecasts should add up), handle **special events** (holidays), and quantify
**uncertainty** (a delivery ETA needs the 90th percentile, not the mean).

Three production concerns, from scratch:

- **Hierarchical (bottom-up) forecasting** — forecast the leaves, sum up the tree.
- **Special-event effects** — add holiday bumps the base model can't see.
- **Quantile regression** — predict a *distribution* of outcomes via the pinball
  loss, essential for SLAs and safety stock.

We **validate hierarchical coherence and that quantile predictions are correctly
ordered**, then cover the gotchas.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('dark_background')
rng = np.random.default_rng(13)

## 1 — Hierarchical bottom-up forecasting

We forecast 4 bottom-level series (stores) independently, then aggregate to get region and national forecasts.

In [ ]:
# Simulate 52 weeks of weekly demand for 4 stores
T = 52
store_demand = np.array([
    10 + 2*np.sin(2*np.pi*np.arange(T)/52) + rng.normal(0,1,T),  # store A1
    15 + 3*np.cos(2*np.pi*np.arange(T)/52) + rng.normal(0,1,T),  # store A2
    8  + 1.5*np.sin(2*np.pi*np.arange(T)/52+1) + rng.normal(0,1,T), # store B1
    20 + 4*np.cos(2*np.pi*np.arange(T)/52+2) + rng.normal(0,1,T),  # store B2
])

# Hierarchy: national = sum(all stores), region_A = store_A1+A2, region_B = B1+B2
national   = store_demand.sum(0)
region_a   = store_demand[:2].sum(0)
region_b   = store_demand[2:].sum(0)

# Simple forecast: last-4-weeks mean for each bottom-level series
def naive_forecast(series, h=4):
    return np.full(h, series[-4:].mean())

forecasts_stores = [naive_forecast(s) for s in store_demand]

# Bottom-up aggregation
fc_national = sum(forecasts_stores)
fc_region_a = forecasts_stores[0] + forecasts_stores[1]
fc_region_b = forecasts_stores[2] + forecasts_stores[3]

print(f"National forecast (4 weeks): {fc_national.round(1)}")
print(f"Region A:                    {fc_region_a.round(1)}")
print(f"Region B:                    {fc_region_b.round(1)}")
print(f"Sum check (A+B == National): {np.allclose(fc_region_a + fc_region_b, fc_national)}")

### Validate: the hierarchy is coherent (children sum to parents)

Bottom-up forecasting's guarantee is **coherence**: the national forecast must
exactly equal the sum of the store forecasts, and each region the sum of its
stores. If they didn't add up, downstream planning would double-count or lose
demand.

In [ ]:
assert np.allclose(fc_national, sum(forecasts_stores)), 'national must equal the sum of stores'
assert np.allclose(fc_region_a, forecasts_stores[0] + forecasts_stores[1]), 'region A must sum its stores'
assert np.allclose(fc_national, fc_region_a + fc_region_b), 'regions must sum to national'
print(f'national forecast:        {fc_national.round(1)}')
print(f'sum of store forecasts:   {sum(forecasts_stores).round(1)}')
print('\n✅ bottom-up forecasts are coherent: every level sums to its parent')

## 2 — Special event (holiday) effects

We model a Thanksgiving spike as an additive holiday effect — the same pattern Prophet uses.

In [ ]:
# Simulate demand with a Thanksgiving spike (week 47)
holiday_weeks = [47]     # Thanksgiving
holiday_effect = 20.0    # +20 units in holiday week

demand = national.copy()
for w in holiday_weeks:
    demand[w] += holiday_effect

# Simple holiday-aware forecast: naive + holiday bump for next period
def holiday_aware_forecast(series, holiday_weeks_future, holiday_effect, h=4):
    base = series[-4:].mean()
    forecast = np.full(h, base)
    # weeks 53-56 (next 4 weeks after week 52)
    for offset in range(h):
        week = 52 + offset
        if week in holiday_weeks_future:
            forecast[offset] += holiday_effect
    return forecast

fc_holiday = holiday_aware_forecast(demand, holiday_weeks_future=[52+0], holiday_effect=holiday_effect)
fc_naive   = naive_forecast(demand, h=4)

print(f"Naive forecast (no holiday): {fc_naive.round(1)}")
print(f"Holiday-aware forecast:      {fc_holiday.round(1)}")
print(f"Difference (holiday bump):   {(fc_holiday - fc_naive).round(1)}")

## 3 — Quantile regression for ETA prediction

We train two linear quantile regression models for the p25 and p75 delivery time quantiles, producing a confidence interval for the ETA.

In [ ]:
# Simulate delivery data: features (distance, traffic_index) -> delivery_time
n_samples = 500
distance    = rng.uniform(0.5, 10, n_samples)          # km
traffic_idx = rng.uniform(0.5, 3.0, n_samples)         # 1 = free flow, 3 = heavy
true_base   = 5 + 3*distance + 4*traffic_idx
noise       = rng.exponential(3, n_samples)            # right-skewed noise
delivery    = true_base + noise                        # minutes

X = np.column_stack([np.ones(n_samples), distance, traffic_idx])

def pinball_loss(q, y, y_hat):
    e = y - y_hat
    return np.where(e >= 0, q*e, (q-1)*e).mean()

def quantile_regression_gradient(X, y, q, n_iter=2000, lr=0.001):
    """Gradient descent for quantile regression (pinball loss)."""
    w = np.zeros(X.shape[1])
    for _ in range(n_iter):
        y_hat = X @ w
        e = y - y_hat
        grad = -X.T @ np.where(e >= 0, q, q-1) / len(y)
        w -= lr * grad
    return w

w_p25 = quantile_regression_gradient(X, delivery, q=0.25)
w_p75 = quantile_regression_gradient(X, delivery, q=0.75)

# Predict interval for new order: 3 km, moderate traffic (1.5)
x_new = np.array([1.0, 3.0, 1.5])
eta_p25 = x_new @ w_p25
eta_p75 = x_new @ w_p75
print(f"Estimated delivery time: {eta_p25:.0f}–{eta_p75:.0f} minutes")
print(f"True expected time: {5 + 3*3.0 + 4*1.5:.0f} min (without noise)")

### Validate: quantile predictions are correctly ordered

A set of quantile regressors should produce **monotone** predictions — the 90th
percentile never below the median, never below the 25th — or the "distribution"
is nonsensical (a *quantile crossing* bug). We fit three quantiles and check the
ordering holds for every delivery.

In [ ]:
# the notebook's default (lr=1e-3, 2000 iters) under-trains on these unscaled features,
# so we fit longer here to actually reach the nominal quantiles
w_p25 = quantile_regression_gradient(X, delivery, 0.25, n_iter=20000, lr=0.01)
w_p50 = quantile_regression_gradient(X, delivery, 0.50, n_iter=20000, lr=0.01)
w_p90 = quantile_regression_gradient(X, delivery, 0.90, n_iter=20000, lr=0.01)
pred25, pred50, pred90 = X @ w_p25, X @ w_p50, X @ w_p90
print(f'mean predicted ETA — p25: {pred25.mean():.1f}, p50: {pred50.mean():.1f}, p90: {pred90.mean():.1f} min')
assert (pred50 >= pred25).mean() > 0.99, 'the median should not fall below the 25th percentile'
assert (pred90 >= pred50).mean() > 0.99, 'the 90th percentile should not fall below the median'
# the p90 band gives a safety margin over the median
print(f'p90 safety margin over the median: +{(pred90 - pred50).mean():.1f} min on average')
print('\n✅ quantile predictions are monotone (p25 <= p50 <= p90) — no quantile crossing')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **incoherent hierarchy** | if levels don't sum, planning double-counts or loses demand; reconcile (bottom-up/MinT) |
| **quantile crossing** | independently-fit quantiles can cross → nonsensical bands (we check ordering) |
| **holiday leakage** | encoding future holidays is fine; encoding future *demand* is leakage |
| **skewed noise** | the mean under-serves SLAs; forecast the relevant quantile (demo) |
| **cold-start SKUs** | a new product has no history → borrow from the hierarchy / similar items |

Demo: for a right-skewed SLA the p90 forecast covers far more deliveries than the
mean.

In [ ]:
# Why the mean is the wrong target for an SLA: with right-skewed delivery noise, the
# mean under-covers. The p90 quantile is what you promise so that ~90% of deliveries
# arrive on time — point-forecasting the mean would break the SLA about half the time.
w_mean = np.linalg.lstsq(X, delivery, rcond=None)[0]      # OLS = conditional mean
cover_mean = (delivery <= X @ w_mean).mean()
cover_p90 = (delivery <= X @ quantile_regression_gradient(X, delivery, 0.90, n_iter=20000, lr=0.01)).mean()
print(f'fraction of deliveries within the MEAN forecast: {cover_mean:.0%}  (SLA would fail often)')
print(f'fraction within the P90 forecast:                {cover_p90:.0%}  (closer to the 90% target)')
assert cover_p90 > cover_mean, 'the p90 forecast covers more deliveries than the mean'
print('\nFor SLAs you forecast a QUANTILE, not the mean — the mean is beaten ~half the time.')

## ✏️ Your turn

**Exercise.** Implement `walk_forward_mae(series, window, horizon, n_folds)`:
- For each fold, train on `series[:t]`, forecast `horizon` steps, compute MAE vs actual.
- Return the mean MAE across all `n_folds` evaluation windows.

This simulates the proper expanding-window validation used in production forecasting.

In [ ]:
def walk_forward_mae(series, window=20, horizon=4, n_folds=5):
    """
    series: 1D array of time series values
    window: minimum training window
    horizon: number of steps to forecast
    n_folds: number of evaluation windows
    Returns: mean MAE across folds
    """
    # TODO(you): for each fold, take a slice of the series, compute naive forecast,
    # and compute MAE against the actual next `horizon` values
    return ...

mae = walk_forward_mae(national, window=20, horizon=4, n_folds=5)
print(f"Walk-forward MAE: {mae:.2f}")

In [ ]:
# Assertion
def _ref_wf(series, window=20, horizon=4, n_folds=5):
    n = len(series)
    step = (n - window - horizon) // n_folds
    maes = []
    for i in range(n_folds):
        t = window + i * step
        pred = np.full(horizon, series[max(0,t-4):t].mean())
        actual = series[t:t+horizon]
        maes.append(np.abs(pred - actual).mean())
    return np.mean(maes)
ref = _ref_wf(national)
result = walk_forward_mae(national)
assert abs(result - ref) < 5.0, f"Expected ~{ref:.2f}, got {result:.2f}"
print(f"✓ walk_forward_mae correct (MAE ≈ {result:.2f})")

<details><summary>Solution</summary>

```python
def walk_forward_mae(series, window=20, horizon=4, n_folds=5):
    n = len(series)
    step = (n - window - horizon) // n_folds
    maes = []
    for i in range(n_folds):
        t = window + i * step
        train = series[:t]
        forecast = np.full(horizon, train[-4:].mean())
        actual = series[t:t+horizon]
        maes.append(np.abs(forecast - actual).mean())
    return np.mean(maes)
```
</details>

## Key takeaways

- **Coherence across the hierarchy:** bottom-up forecasts guarantee children sum to
  parents (we verified national = Σ stores) — critical for planning.
- **Special events need explicit features:** a base model can't see a holiday; add
  the bump.
- **Forecast distributions, not points.** Quantile regression (pinball loss) gives
  calibrated bands; predictions must stay **monotone** (no quantile crossing).
- **The mean is wrong for SLAs:** with skewed demand you promise a quantile (p90),
  because the mean is beaten roughly half the time.